# GRN_01 — SERGIO Simulation and Ground-Truth Preparation

This notebook has the following workflow:

1. Generate a three-layer regulatory network (master regulators → transcription factors → target genes).
2. Write SERGIO's `targets_file.txt` and `master_regulators_file.txt` inputs.
3. Run SERGIO to simulate gene expression.
4. Save the expression matrix and a binary TF → target-gene ground-truth matrix.

## Small, Colab-friendly starting configuration

- **5 master regulators, 10 TFs, 15 target genes** (30 genes total).
- **1 bin with 120 cells**. This is much smaller than 100 bins with one cell per bin and produces 120 expression columns.
- The label matrix contains every TF–target-gene candidate pair: `1` means the edge exists in the generated network; `0` means it does not.
- Both Python's `random` and NumPy are seeded for reproducibility.

The TF → target label matrix has shape `(n_target_genes, n_TFs)`. The expression CSV contains all genes (including master regulators), with genes as rows and simulated cells as columns.

## 1. Install and import SERGIO

Run this cell in Google Colab. It installs `python-igraph` and clones SERGIO only if the repository is not already present.

In [1]:
%pip -q install python-igraph

import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/"
if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = os.getcwd()

SERGIO_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/SERGIO")

if not SERGIO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/PayamDiba/SERGIO.git", str(SERGIO_DIR)],
        check=True
    )

if str(SERGIO_DIR) not in sys.path:
    sys.path.append(str(SERGIO_DIR))

print("SERGIO repository:", SERGIO_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 127.9 MB/s eta 0:00:00
Mounted at /content/drive/
SERGIO repository: /content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/SERGIO


## 2. Set seeds and define the network

The structure follows the original notebook, but uses fewer genes so the first run is quick. Each TF is regulated by two master regulators; each target gene is regulated by two to four TFs. Interaction strengths are sampled from `[1, 5]`, with a 20% chance of a negative sign. Hill coefficients are 2 with probability 0.9 and 1 otherwise.

In [6]:
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Network structure
n_MR = 25
n_TF = 50
n_G = 100
total_genes = n_MR + n_TF + n_G

# Simulation design: one bin, multiple cells in that bin
n_bins = 1
n_cells_per_bin = 300

targets_filename = "targets_file.txt"
mr_filename = "master_regulators_file.txt"

print(f"Genes total: {total_genes} (MR={n_MR}, TF={n_TF}, targets={n_G})")
print(f"Simulation: {n_bins} bin × {n_cells_per_bin} cells")

Genes total: 175 (MR=25, TF=50, targets=100)
Simulation: 1 bin × 300 cells


## 3. Generate SERGIO's target/network file

SERGIO expects each target node's line to contain the target ID, number of regulators, regulator IDs, interaction strengths, and Hill coefficients. Node IDs are contiguous: master regulators first, then TFs, then target genes.

In [7]:
with open(targets_filename, "w") as f:
    # Layer 1 -> Layer 2: master regulators regulate TFs
    d_mr_tf = 2
    for tf_id in range(n_MR, n_MR + n_TF):
        regs = random.sample(range(0, n_MR), d_mr_tf)
        ks = [
            np.random.uniform(1.0, 5.0) * (-1 if random.random() < 0.2 else 1)
            for _ in range(d_mr_tf)
        ]
        hills = [2 if random.random() < 0.9 else 1 for _ in range(d_mr_tf)]
        line = (
            f"{tf_id},{d_mr_tf},"
            + ",".join(map(str, regs)) + ","
            + ",".join(map(str, ks)) + ","
            + ",".join(map(str, hills)) + "\n"
        )
        f.write(line)

    # Layer 2 -> Layer 3: TFs regulate target genes
    for target_id in range(n_MR + n_TF, total_genes):
        d_tf_g = random.randint(2, 4)
        regs = random.sample(range(n_MR, n_MR + n_TF), d_tf_g)
        ks = [
            np.random.uniform(1.0, 5.0) * (-1 if random.random() < 0.2 else 1)
            for _ in range(d_tf_g)
        ]
        hills = [2 if random.random() < 0.9 else 1 for _ in range(d_tf_g)]
        line = (
            f"{target_id},{d_tf_g},"
            + ",".join(map(str, regs)) + ","
            + ",".join(map(str, ks)) + ","
            + ",".join(map(str, hills)) + "\n"
        )
        f.write(line)

print(f"Wrote {targets_filename}")
print(Path(targets_filename).read_text().splitlines()[:3])

Wrote targets_file.txt
['25,2,20,3,-2.49816047538945,4.802857225639665,2,2', '26,2,21,23,3.9279757672456204,-3.3946339367881464,2,2', '27,2,6,7,1.624074561769746,-1.6239780813448106,2,2']


## 4. Generate master-regulator production rates

Because this setup uses one bin, the master-regulator file has one production rate per master regulator. The simulator then generates multiple cells within that bin. If you later increase `n_bins`, generate one rate per bin on each row.

In [8]:
low_range = (0.1, 0.5)
high_range = (1.5, 3.0)

with open(mr_filename, "w") as f:
    for mr_id in range(n_MR):
        # One rate per bin; n_bins=1 here.
        rates = []
        for _ in range(n_bins):
            if random.random() < 0.5:
                rate = np.random.uniform(*low_range)
            else:
                rate = np.random.uniform(*high_range)
            rates.append(rate)
        f.write(f"{mr_id}," + ",".join(map(str, rates)) + "\n")

print(f"Wrote {mr_filename}")
print(Path(mr_filename).read_text())

Wrote master_regulators_file.txt
0,2.7396861991616124
1,1.9800744015459175
2,2.8432848427443007
3,0.25568067149366525
4,1.5162564772204474
5,0.46215279056770553
6,0.13651467071445345
7,0.22772545503616595
8,0.480024786820322
9,2.925910720406334
10,0.3293751552493145
11,0.3527348848679197
12,2.1726682829674795
13,0.21728430867922582
14,0.2314658181479664
15,2.5087776841155573
16,2.62856179415652
17,2.687368565588773
18,0.41584725711782156
19,0.13648244121947617
20,2.2416304570538723
21,0.12302350400665772
22,0.31981155292949426
23,2.1622957520600656
24,2.8315562741374496



## 5. Patch deprecated NumPy aliases if needed, then run SERGIO

The patch is limited to the cloned SERGIO source file and handles the legacy `np.int(...)` / `np.float(...)` calls present in some versions. The simulation cell is kept separate so that runtime problems can be localized.

In [9]:
sergio_source = SERGIO_DIR / "SERGIO" / "sergio.py"
if not sergio_source.exists():
    raise FileNotFoundError(
        f"Could not find SERGIO source at {sergio_source}. "
        "Check the cloned repository structure."
    )

source = sergio_source.read_text()
patched_source = source.replace("np.int(", "int(").replace("np.float(", "float(")
if patched_source != source:
    sergio_source.write_text(patched_source)
    print("Applied compatibility patch to deprecated NumPy aliases.")
else:
    print("No deprecated np.int(/np.float( calls found; no patch needed.")

# Force a fresh import in case the module was already loaded.
for module_name in ["SERGIO.sergio", "sergio"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from SERGIO.sergio import sergio

sim = sergio(
    number_genes=total_genes,
    number_bins=n_bins,
    number_sc=n_cells_per_bin,
    noise_params=1,
    noise_type="dpd",
    decays=0.8
)

print("Building regulatory graph...")
sim.build_graph(targets_filename, mr_filename, shared_coop_state=2)

print("Running SERGIO simulation...")
sim.simulate()

expr_matrix_list = sim.getExpressions()
expr_matrix = np.concatenate(expr_matrix_list, axis=1)

print("Simulation complete.")
print("Raw expression matrix shape:", expr_matrix.shape)
print("Expected gene count:", total_genes)
print("Expected cell count:", n_bins * n_cells_per_bin)

assert expr_matrix.shape[0] == total_genes, (
    f"Expected {total_genes} gene rows, got {expr_matrix.shape[0]}"
)
assert expr_matrix.shape[1] == n_bins * n_cells_per_bin, (
    f"Expected {n_bins * n_cells_per_bin} cell columns, got {expr_matrix.shape[1]}"
)

No deprecated np.int(/np.float( calls found; no patch needed.
Building regulatory graph...
Running SERGIO simulation...
Start simulating new level
There are 25 genes to simulate in this layer
Done with current level
Start simulating new level
There are 50 genes to simulate in this layer
Done with current level
Start simulating new level
There are 100 genes to simulate in this layer
Done with current level
Simulation complete.
Raw expression matrix shape: (175, 300)
Expected gene count: 175
Expected cell count: 300


## 6. Save the expression matrix as CSV

Rows are all simulated genes in SERGIO node order. Columns are cells. The row names make the three network layers explicit.

In [10]:
gene_labels = (
    [f"MR_{i}" for i in range(n_MR)]
    + [f"TF_{i}" for i in range(n_TF)]
    + [f"TargetGene_{i}" for i in range(n_G)]
)
cell_labels = [f"Cell_{i}" for i in range(expr_matrix.shape[1])]

df_expr = pd.DataFrame(expr_matrix, index=gene_labels, columns=cell_labels)
expression_csv = "simulated_expression_matrix.csv"
df_expr.to_csv(expression_csv, index_label="gene")

print(f"Saved expression matrix: {expression_csv}")
print("Shape:", df_expr.shape)
display(df_expr.iloc[:5, :5])

Saved expression matrix: simulated_expression_matrix.csv
Shape: (175, 300)


,Cell_0,Cell_1,Cell_2,Cell_3,Cell_4
MR_0,2.257744,1.639490,1.452429,3.715810,1.670728
MR_1,2.352713,1.856065,4.818385,3.810054,2.053257
MR_2,1.912773,5.280657,4.193169,3.113855,2.998089
MR_3,0.301949,0.547726,1.478284,0.106580,0.620141
MR_4,0.701835,2.389822,1.838029,2.254454,1.516288


## 7. Build the TF → target-gene binary ground-truth matrix

This matrix is aligned to the intended GRN inference task: rows are the 15 target genes, columns are the 10 TFs, and each cell indicates whether the TF regulates that target in the generated network. Thus **all `n_G × n_TF` candidate pairs are represented**, including non-edges as zero labels. Master-regulator → TF edges are intentionally excluded from this label matrix.

In [11]:
ground_truth_matrix = np.zeros((n_G, n_TF), dtype=np.int64)

with open(targets_filename, "r") as f:
    for line in f:
        parts = line.strip().split(",")
        target_node_id = int(parts[0])
        num_regulators = int(parts[1])
        regulator_ids = [int(x) for x in parts[2:2 + num_regulators]]

        # Only target genes (Layer 3) are rows in this matrix.
        if n_MR + n_TF <= target_node_id < total_genes:
            target_idx = target_node_id - (n_MR + n_TF)
            for reg_id in regulator_ids:
                # Include only TF -> target edges, not MR -> target edges.
                if n_MR <= reg_id < n_MR + n_TF:
                    tf_idx = reg_id - n_MR
                    ground_truth_matrix[target_idx, tf_idx] = 1

target_names = [f"TargetGene_{i}" for i in range(n_G)]
tf_names = [f"TF_{i}" for i in range(n_TF)]
df_labels = pd.DataFrame(ground_truth_matrix, index=target_names, columns=tf_names)

labels_csv = "ground_truth_labels.csv"
df_labels.to_csv(labels_csv, index_label="target_gene")

print(f"Saved ground-truth label matrix: {labels_csv}")
print("Shape:", df_labels.shape)
print("Positive TF → target edges:", int(df_labels.to_numpy().sum()))
print("Candidate TF-target pairs:", int(df_labels.size))
display(df_labels.iloc[:5, :])

Saved ground-truth label matrix: ground_truth_labels.csv
Shape: (100, 50)
Positive TF → target edges: 303
Candidate TF-target pairs: 5000


,TF_0,TF_1,TF_2,TF_3,TF_4,TF_5,TF_6,TF_7,TF_8,TF_9,...,TF_40,TF_41,TF_42,TF_43,TF_44,TF_45,TF_46,TF_47,TF_48,TF_49
TargetGene_0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TargetGene_1,0,0,0,0,0,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
TargetGene_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TargetGene_3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
TargetGene_4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## 8. Final checks and download

The two CSV files below are the requested outputs. The expression matrix has all genes × cells; the label matrix has target genes × candidate TFs. A zero in the label matrix means the candidate pair was not included as an edge in the generated network—it does not establish that the biological relationship is absent in nature.

In [12]:
from pathlib import Path

for filename in [expression_csv, labels_csv]:
    p = Path(filename)
    print(f"{p.name}: {p.resolve()} ({p.stat().st_size:,} bytes)")

assert df_expr.shape == (total_genes, n_bins * n_cells_per_bin)
assert df_labels.shape == (n_G, n_TF)
assert set(np.unique(df_labels.to_numpy())).issubset({0, 1})

print("\nAll shape and binary-label checks passed.")

simulated_expression_matrix.csv: /content/simulated_expression_matrix.csv (966,788 bytes)
ground_truth_labels.csv: /content/ground_truth_labels.csv (11,692 bytes)

All shape and binary-label checks passed.


## 9. Prepare pair-level data for GRN_03

In [13]:
data_dir = os.path.join(PROJECT_DIR, "data")
os.makedirs(data_dir, exist_ok=True)

# Start with the simulated expression matrix: genes × cells.
expr = np.asarray(expr_matrix, dtype=np.float32).copy()
expr = np.nan_to_num(expr, nan=0.0, posinf=0.0, neginf=0.0)

# Match the preprocessing used by grn_experiment_utils.make_pair_sequences.
if np.min(expr) >= 0 and np.percentile(expr, 99) > 20:
    expr = np.log1p(expr)

# Standardise each gene across cells.
mu = expr.mean(axis=1, keepdims=True)
sd = expr.std(axis=1, keepdims=True)
expr = (expr - mu) / np.maximum(sd, 1e-5)

# GRN_03's Transformer uses condition sequences.
# Cap the number of cells/conditions at 64 for consistency with the utility.
rng = np.random.default_rng(SEED)
max_conditions = 64

if expr.shape[1] > max_conditions:
    condition_idx = np.sort(
        rng.choice(expr.shape[1], size=max_conditions, replace=False)
    )
    expr = expr[:, condition_idx]

# Candidate regulator and target indices in the full expression matrix.
tf_gene_indices = np.arange(n_MR, n_MR + n_TF)
target_gene_indices = np.arange(n_MR + n_TF, total_genes)

# One example for every TF → target pair.
# Each example has shape (conditions, 2):
# [TF expression, target-gene expression].
X_list, y_list, pair_ids = [], [], []

for target_idx, target_name in enumerate(target_names):
    for tf_idx, tf_name in enumerate(tf_names):
        tf_gene_idx = n_MR + tf_idx
        target_gene_idx = n_MR + n_TF + target_idx

        X_list.append(
            np.stack(
                [expr[tf_gene_idx], expr[target_gene_idx]],
                axis=-1
            )
        )
        y_list.append(ground_truth_matrix[target_idx, tf_idx])
        pair_ids.append(f"{tf_name}->{target_name}")

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.float32)
pair_ids = np.asarray(pair_ids)

# Save in the exact location and with the keys expected by GRN_03.
out = os.path.join(data_dir, "sergio_pair_data.npz")

np.savez_compressed(
    out,
    X=X,
    y=y,
    pair_ids=pair_ids,
    expression=expr_matrix,
    edge_list=np.asarray(
        [
            (n_MR + tf_idx, n_MR + n_TF + target_idx)
            for target_idx, tf_idx in zip(*np.where(ground_truth_matrix == 1))
        ],
        dtype=int
    ).reshape(-1, 2)
)

print("Saved:", out)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive edges:", int(y.sum()))
print("Candidate pairs:", len(y))
print("Positive prevalence:", float(y.mean()))

assert X.shape == (n_TF * n_G, expr.shape[1], 2)
assert y.shape == (n_TF * n_G,)
assert int(y.sum()) == int(ground_truth_matrix.sum())

Saved: /content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/data/sergio_pair_data.npz
X shape: (5000, 64, 2)
y shape: (5000,)
Positive edges: 303
Candidate pairs: 5000
Positive prevalence: 0.060600001364946365
